# Exercise 1 - KNN on MNIST dataset

In [7]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tarfile
from io import BytesIO
from urllib.request import urlopen

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

knn = KNeighborsClassifier()
param_grid = {"n_neighbors":[3,4,5], "weights":["uniform","distance"]}

grid = GridSearchCV(knn, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))

Best parameters: {'n_neighbors': 3, 'weights': 'uniform'}
Best CV accuracy: nan
Test accuracy: 0.9705


# Exercise 2 - KNN on augmented MNIST dataset

In [3]:
def shift_images(X, dx, dy):
    images = X.reshape(-1, 28, 28)
    shifted = np.roll(images, shift=(dy, dx), axis=(1, 2))

    if dy > 0:
        shifted[:, :dy, :] = 0
    elif dy < 0:
        shifted[:, dy:, :] = 0

    if dx > 0:
        shifted[:, :, :dx] = 0
    elif dx < 0:
        shifted[:, :, dx:] = 0

    return shifted.reshape(-1, 784)

X_left  = shift_images(X_train, -1, 0)
X_right = shift_images(X_train, 1, 0)
X_up    = shift_images(X_train, 0, -1)
X_down  = shift_images(X_train, 0, 1)

X_train_aug = np.concatenate([X_train, X_left, X_right, X_up, X_down])
y_train_aug = np.tile(y_train, 5)

print("Original:", X_train.shape)
print("Augmented:", X_train_aug.shape)

Original: (60000, 784)
Augmented: (300000, 784)


In [4]:
best_knn.fit(X_train_aug, y_train_aug)
y_pred_aug = best_knn.predict(X_test)
print("Test accuracy after augmentation:", accuracy_score(y_test, y_pred_aug))

Test accuracy after augmentation: 0.9756


The accuracy slightly increases for the dataset with data augmented/training set explanded (4 extra images per image with each image shifted by one pixel in each direction).

# Exercise 3 - Classification on the Titanic dataset

In [8]:
url = "https://github.com/ageron/data/raw/main/titanic.tgz"

with tarfile.open(fileobj=BytesIO(urlopen(url).read()), mode="r:gz") as tar:
    train_data = pd.read_csv(tar.extractfile("titanic/train.csv"))
    test_data = pd.read_csv(tar.extractfile("titanic/test.csv"))

print("Training shape:", train_data.shape)
print("Test shape:", test_data.shape)

train_data.head()

Training shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
X_train = train_data.drop("Survived", axis=1)
y_train = train_data["Survived"]

num_attribs = ["Age", "SibSp", "Parch", "Fare"]
cat_attribs = ["Pclass", "Sex", "Embarked"]

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
preprocessing = ColumnTransformer([("num", num_pipeline, num_attribs), ("cat", cat_pipeline, cat_attribs)])
model = make_pipeline(preprocessing, RandomForestClassifier(n_estimators=100, random_state=42))
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

print("=" * 60)
print("TITANIC CLASSIFIER")
print("=" * 60)
print(f"CV accuracy: {scores.mean():.4f}")
print(f"CV std:      {scores.std():.4f}")

TITANIC CLASSIFIER
CV accuracy: 0.8059
CV std:      0.0326


A Random Forest classifier was trained on the Titanic dataset using a preprocessing pipeline to handle missing values, scale numerical features, and encode categorical features. Five-fold cross-validation was used to evaluate the model, providing an estimate of its generalization performance. We got a decent accuracy and a very small CV error suggesting model fits well on unseen data.